In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import random
import copy
import gc
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report

DIR_BASE = '/kaggle/input/embeddings-tp2-bilstm'

# Semilla para reproducibilidad
SEED = 33
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entorno Kaggle listo. Usando: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Constantes
COLS_X = 772 # 768 de embeddings + 4
DTYPE_X = np.float16

class OracionesDataset(Dataset):
    def __init__(self, dir_base, split='train'):
        self.split = split

        path_X = os.path.join(dir_base, f'dataset_{split}_completo.dat')
        path_n_sent = os.path.join(dir_base, f'n_sentence_{split}.npy')

        # Ajustamos prefijos según archivos
        prefix = 'y_train' if split == 'train' else 'y_eval'
        path_y_cap = os.path.join(dir_base, f'{prefix}_cap.npy')
        path_y_ini = os.path.join(dir_base, f'{prefix}_ini.npy')
        path_y_fin = os.path.join(dir_base, f'{prefix}_fin.npy')

        # Memmap (Lectura eficiente)
        filesize = os.path.getsize(path_X)
        n_rows = filesize // (COLS_X * np.dtype(DTYPE_X).itemsize)
        self.X = np.memmap(path_X, dtype=DTYPE_X, mode='r', shape=(n_rows, COLS_X))

        # Cargar índices en RAM
        print(f"[{split.upper()}] Cargando índices...")
        self.n_sentence = np.load(path_n_sent)
        self.y_cap = np.load(path_y_cap)
        self.y_ini = np.load(path_y_ini)
        self.y_fin = np.load(path_y_fin)

        _, start_indices, lengths = np.unique(self.n_sentence, return_index=True, return_counts=True)
        self.starts = start_indices
        self.lengths = lengths
        self.num_sentences = len(start_indices)

    def __len__(self):
        return self.num_sentences

    def __getitem__(self, idx):
        start = self.starts[idx]
        real_len = self.lengths[idx]

        # Recorte de seguridad (Aunque en Kaggle hay memoria, ayuda a la velocidad)
        MAX_LEN = 250
        length = min(real_len, MAX_LEN)
        end = start + length

        x_seq = torch.tensor(self.X[start:end], dtype=torch.float32)

        target_list = []
        # Acceso directo a arrays de numpy en RAM
        raw_c = self.y_cap[start:end]
        raw_i = self.y_ini[start:end]
        raw_f = self.y_fin[start:end]

        for i in range(length):
            # 1. Puntuación Inicial
            v_ini = [1.0] if (str(raw_i[i]) not in ['', 'nan']) else [0.0]

            # 2. Puntuación Final
            v_fin = [0.0]*4
            opts_f = ['', '.', ',', '?']
            val_f = str(raw_f[i])
            if val_f in opts_f: v_fin[opts_f.index(val_f)] = 1.0
            else: v_fin[0] = 1.0

            # 3. Capitalización
            v_cap = [0.0]*4
            val_c = int(raw_c[i])
            if 0 <= val_c <= 3: v_cap[val_c] = 1.0

            target_list.append(v_ini + v_fin + v_cap)

        y_seq = torch.tensor(target_list, dtype=torch.float32)
        return x_seq, y_seq

def collate_pad(batch):
    xx, yy = zip(*batch)
    x_pad = pad_sequence(xx, batch_first=True, padding_value=0)
    y_pad = pad_sequence(yy, batch_first=True, padding_value=0)
    return x_pad, y_pad

# --- CARGA ---
print("--- Instanciando Datasets ---")
train_ds = OracionesDataset(DIR_BASE, split='train')
eval_ds  = OracionesDataset(DIR_BASE, split='eval')

# BATCH SIZE
BATCH_SIZE = 512

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_pad, pin_memory=True, num_workers=0
)
eval_loader  = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_pad, pin_memory=True, num_workers=0
)

print(f"DataLoaders listos. Batch Size: {BATCH_SIZE}")

In [ ]:
class NuestraLSTM_Bi(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(NuestraLSTM_Bi, self).__init__()

        # LSTM BIDIRECCIONAL
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_prob if num_layers > 1 else 0,
            bidirectional=True # <---
        )

        self.dropout = nn.Dropout(dropout_prob)

        # La salida de una BiLSTM es el doble de grande (Forward + Backward)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits

# Loss Function (Igual que para la unidireccional)
class MixedLoss(nn.Module):
    def __init__(self, w_bin, w_soft1, w_soft2):
        super(MixedLoss, self).__init__()
        self.loss_bin = nn.BCEWithLogitsLoss(reduction='none')
        self.loss_mc = nn.CrossEntropyLoss(reduction='none')
        self.weights = (w_bin, w_soft1, w_soft2)

    def forward(self, logits, targets):
        mask = (targets.sum(dim=-1) != 0).float()

        l_bin, l_s1, l_s2 = logits[..., 0], logits[..., 1:5], logits[..., 5:9]
        t_bin = targets[..., 0]
        t_s1 = targets[..., 1:5].argmax(dim=-1)
        t_s2 = targets[..., 5:9].argmax(dim=-1)

        loss_b = self.loss_bin(l_bin, t_bin) * mask
        loss_m1 = self.loss_mc(l_s1.permute(0,2,1), t_s1) * mask
        loss_m2 = self.loss_mc(l_s2.permute(0,2,1), t_s2) * mask

        N = mask.sum() + 1e-8
        return (self.weights[0]*loss_b.sum()/N +
                self.weights[1]*loss_m1.sum()/N +
                self.weights[2]*loss_m2.sum()/N)

print("Modelo BiLSTM definido.")

In [ ]:
# Grilla de Hiperparámetros
param_grid = {
    'hidden_size': [128, 256, 512],
    'num_layers':  [1, 2, 3],
    'dropout':     [0.3, 0.5],
    'lr':          [0.001, 0.0005],
    'loss_weights': [
        (1.0, 3.0, 1.0), # Enfasis Fuerte
        (1.0, 5.0, 1.0), # Enfasis Muy Fuerte
        (1.0, 1.0, 1.0)  # Balanceado (Control)
    ]
}

NUM_TRIALS = 10
EPOCHS_PER_TRIAL = 10
PATIENCE = 2           # Si en 2 épocas no mejora, corta

print(f"Random Search configurado: {NUM_TRIALS} combinaciones distintas.")
print(f"Total épocas máx: {NUM_TRIALS * EPOCHS_PER_TRIAL}")

In [ ]:
# ENTRENAMIENTO
best_global_f1 = 0.0
best_global_params = {}
best_model_state = None
scaler = GradScaler()

print(f"INICIANDO BÚSQUEDA EN: {device}")

for trial in range(NUM_TRIALS):
    params = {k: random.choice(v) for k, v in param_grid.items()}
    w_bin, w_s1, w_s2 = params['loss_weights']

    print(f"\n{'='*60}")
    print(f"TRIAL {trial+1}/{NUM_TRIALS} | Params: {params}")

    model = NuestraLSTM_Bi( # <--- Usamos la clase BiLSTM
        input_size=772,
        hidden_size=params['hidden_size'],
        num_layers=params['num_layers'],
        output_size=9,
        dropout_prob=params['dropout']
    ).to(device)

    criterion = MixedLoss(w_bin, w_s1, w_s2)
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])

    best_trial_f1 = 0.0
    patience_count = 0

    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)

        for x_b, y_b in pbar:
            x_b, y_b = x_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast(): # Mixed Precision para ahorrar VRAM y ganar velocidad
                logits = model(x_b)
                loss = criterion(logits, y_b)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
            del x_b, y_b, logits, loss # Limpieza inmediata

        avg_train_loss = train_loss / len(train_loader)
        torch.cuda.empty_cache()

        # Eval
        model.eval()
        all_preds, all_targs = [], []
        with torch.no_grad():
            for x_b, y_b in eval_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                with autocast():
                    logits = model(x_b)
                    preds = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)

                targs = y_b[..., 1:5].argmax(dim=-1)
                mask = (y_b.sum(dim=-1) != 0)
                all_preds.extend(preds[mask].cpu().numpy())
                all_targs.extend(targs[mask].cpu().numpy())
                del x_b, y_b, logits, preds, targs, mask

        val_f1 = f1_score(all_targs, all_preds, average='macro', zero_division=0)
        print(f"   Ep {epoch+1}: Loss {avg_train_loss:.4f} | Val F1 (Punt): {val_f1:.4f}")

        if val_f1 > best_trial_f1:
            best_trial_f1 = val_f1
            patience_count = 0
            if val_f1 > best_global_f1:
                print(f"      🏆 NUEVO RÉCORD: {val_f1:.4f}")
                best_global_f1 = val_f1
                best_global_params = params
                best_model_state = copy.deepcopy(model.state_dict())

                # Guardamos checkpoint en Kaggle Working por si crashea la sesión
                torch.save(best_model_state, '/kaggle/working/checkpoint_best.pth')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print("      Early Stopping.")
                break

    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#"*50)
print(f"MEJOR F1 GLOBAL: {best_global_f1:.4f}")
print(f"MEJORES PARAMS: {best_global_params}")

# Guardado Final
if best_model_state:
    path_save = '/kaggle/working/mejor_modelo_bilstm.pth'
    torch.save({
        'state': best_model_state,
        'params': best_global_params,
        'f1': best_global_f1
    }, path_save)
    print(f"💾 Modelo final guardado en: {path_save}")
    print("Recuerda descargar este archivo desde la pestaña 'Output' a la derecha.")

In [ ]:
BEST_PARAMS = {
    'hidden_size': 256,
    'num_layers': 3,
    'dropout': 0.3,
    'output_size': 9
}

MODEL_PATH = '/kaggle/input/modelo-bilstm-entrenado/mejor_modelo_bilstm.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NOMBRES_FIN = ['VACIO', 'PUNTO (.)', 'COMA (,)', 'SIGNO (?)']
NOMBRES_CAP = ['MINUSCULA', 'TITLE (Mayus)', 'MEZCLA', 'MAYUSCULA']
NOMBRES_INI = ['NO', 'SI (¿)']

# Instanciamos la arquitectura BIDIRECCIONAL
model = NuestraLSTM_Bi(
    input_size=772,
    hidden_size=BEST_PARAMS['hidden_size'],
    num_layers=BEST_PARAMS['num_layers'],
    output_size=BEST_PARAMS['output_size'],
    dropout_prob=BEST_PARAMS['dropout']
).to(device)

# Cargar pesos
checkpoint = torch.load(MODEL_PATH, weights_only=False)
if 'state' in checkpoint:
    model.load_state_dict(checkpoint['state'])
else:
    model.load_state_dict(checkpoint)

model.eval()

# --- 3. EVALUACIÓN DETALLADA ---

true_fin, pred_fin = [], []
true_cap, pred_cap = [], []
true_ini, pred_ini = [], []

with torch.no_grad():
    for x_b, y_b in tqdm(eval_loader, desc="Evaluando"):
        x_b = x_b.to(device)

        # Forward
        logits = model(x_b)

        # --- DECODIFICACIÓN ---

        # 1. Puntuación Final (Indices 1-4)
        p_fin = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)
        t_fin = y_b[..., 1:5].argmax(dim=-1)

        # 2. Capitalización (Indices 5-8)
        p_cap = torch.softmax(logits[..., 5:9], dim=-1).argmax(dim=-1)
        t_cap = y_b[..., 5:9].argmax(dim=-1)

        # 3. Puntuación Inicial (Indice 0 - Binario)
        p_ini = (torch.sigmoid(logits[..., 0]) > 0.5).float()
        t_ini = y_b[..., 0]

        # --- FILTRAR PADDING ---
        mask = (y_b.sum(dim=-1) != 0).cpu()

        true_fin.extend(t_fin.cpu()[mask].numpy())
        pred_fin.extend(p_fin.cpu()[mask].numpy())

        true_cap.extend(t_cap.cpu()[mask].numpy())
        pred_cap.extend(p_cap.cpu()[mask].numpy())

        true_ini.extend(t_ini.cpu()[mask].numpy())
        pred_ini.extend(p_ini.cpu()[mask].numpy())

# --- 4. REPORTES ---
print("\n" + "★"*60)
print(f" RESULTADOS DETALLADOS - BiLSTM (F1 Global: {checkpoint.get('f1', 'N/A'):.4f})")
print("★"*60)

print("\n TAREA 1: PUNTUACIÓN FINAL (.,?)")
print(classification_report(true_fin, pred_fin, target_names=NOMBRES_FIN, digits=4))

print("\n TAREA 2: CAPITALIZACIÓN")
print(classification_report(true_cap, pred_cap, target_names=NOMBRES_CAP, digits=4))

print("\n TAREA 3: PUNTUACIÓN INICIAL (¿)")
print(classification_report(true_ini, pred_ini, target_names=NOMBRES_INI, digits=4))

In [ ]:
from torch.cuda.amp import GradScaler

PARAMS = {
    'hidden_size': 256,
    'num_layers': 3,
    'dropout': 0.3,
    'lr': 0.0005,
    'loss_weights': (1.0, 3.0, 1.0)
}

# Rutas
MODEL_PATH_ORIGINAL = '/kaggle/input/modelo-bilstm-entrenado/mejor_modelo_bilstm.pth'
MODEL_PATH_FINAL    = '/kaggle/working/modelo_bilstm_final_v2.pth'

EXTRA_EPOCHS = 20
PATIENCE = 4
BATCH_SIZE = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Iniciando Fine-Tuning Extendido en: {device}")

# --- 3. RECONSTRUIR DATALOADERS ---
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_pad, pin_memory=True, num_workers=0
)
eval_loader  = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_pad, pin_memory=True, num_workers=0
)

# --- 4. CARGAR MODELO PREVIO ---
model = NuestraLSTM_Bi(
    input_size=772,
    hidden_size=PARAMS['hidden_size'],
    num_layers=PARAMS['num_layers'],
    output_size=9,
    dropout_prob=PARAMS['dropout']
).to(device)

print(f"🔄 Cargando pesos base desde: {MODEL_PATH_ORIGINAL}")
checkpoint = torch.load(MODEL_PATH_ORIGINAL, weights_only=False)

if 'state' in checkpoint:
    model.load_state_dict(checkpoint['state'])
    best_f1 = checkpoint.get('f1', 0.8374)
else:
    model.load_state_dict(checkpoint)
    best_f1 = 0.8374

print(f"✅ Pesos cargados. F1 Base: {best_f1:.4f}")

# --- 5. BUCLE DE ENTRENAMIENTO ---
criterion = MixedLoss(*PARAMS['loss_weights'])
optimizer = optim.Adam(model.parameters(), lr=PARAMS['lr'])
scaler = GradScaler()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=1
)

best_model_state = copy.deepcopy(model.state_dict())
patience_counter = 0

print(f"🏁 Entrenando por {EXTRA_EPOCHS} épocas más...")

for epoch in range(EXTRA_EPOCHS):
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)

    for x_b, y_b in pbar:
        x_b, y_b = x_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(x_b)
            loss = criterion(logits, y_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        del x_b, y_b, logits, loss

    avg_train_loss = train_loss / len(train_loader)
    torch.cuda.empty_cache()

    # Eval
    model.eval()
    all_preds, all_targs = [], []
    with torch.no_grad():
        for x_b, y_b in eval_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(x_b)
                preds = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)

            targs = y_b[..., 1:5].argmax(dim=-1)
            mask = (y_b.sum(dim=-1) != 0)
            all_preds.extend(preds[mask].cpu().numpy())
            all_targs.extend(targs[mask].cpu().numpy())
            del x_b, y_b, logits, preds, targs, mask

    val_f1 = f1_score(all_targs, all_preds, average='macro', zero_division=0)

    scheduler.step(val_f1)
    lr_actual = optimizer.param_groups[0]['lr']

    print(f"   Ep {epoch+1}: Loss {avg_train_loss:.4f} | Val F1: {val_f1:.4f} | LR: {lr_actual:.1e}")

    if val_f1 > best_f1:
        print(f" MEJORA: {best_f1:.4f} -> {val_f1:.4f}")
        best_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0

        torch.save({
            'state': best_model_state,
            'params': PARAMS,
            'f1': best_f1
        }, MODEL_PATH_FINAL)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(" Early Stopping (Ya no mejora).")
            break

    gc.collect()

print("\n" + "="*50)
print(f"ENTRENAMIENTO FINALIZADO. MEJOR F1: {best_f1:.4f}")
print(f"Modelo guardado en: {MODEL_PATH_FINAL}")